In [1]:
!pip install -q chromadb langchain langchain-community langchain-ollama pypdf sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.

In [2]:
!apt-get update -qq
!apt-get install -y -qq zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
!ollama --version

In [5]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

print("Ollama server started")

Ollama server started


In [6]:
!ollama pull llama3.2:3b

In [7]:
!ollama list

NAME           ID              SIZE      MODIFIED               
llama3.2:3b    a80c4f17acd5    2.0 GB    Less than a second ago    


In [8]:
import chromadb

client = chromadb.Client()

collection = client.create_collection(
    name="w6d4_rag_collection",
    metadata={"hnsw:space": "cosine"}
)

print("ChromaDB collection created successfully")

ChromaDB collection created successfully


In [9]:
documents = [
    "Artificial intelligence enables machines to perform tasks that normally require human intelligence.",
    "Machine learning allows computers to learn patterns from data.",
    "Deep learning uses neural networks with multiple layers.",
    "Natural language processing helps computers understand human language.",
    "Large language models can generate and understand text.",
    "Retrieval augmented generation combines information retrieval with language generation.",
    "Vector databases store numerical representations of data.",
    "Embeddings represent text as numerical vectors.",
    "ChromaDB is a vector database designed for AI applications.",
    "Cosine similarity measures the similarity between two vectors.",
    "Metadata can be used to filter documents during retrieval.",
    "LangChain provides tools for building applications with language models.",
    "Ollama allows language models to run locally.",
    "RAG systems retrieve relevant context before generating an answer.",
    "Document chunking divides large documents into smaller pieces.",
    "Semantic search finds documents based on meaning rather than exact keywords.",
    "AI applications can use embeddings for efficient information retrieval.",
    "A vector store can perform similarity searches over embedded documents.",
    "Prompt context helps language models generate more relevant answers.",
    "RAG can reduce the need for a language model to rely only on its training data."
]

print("Number of documents:", len(documents))

Number of documents: 20


In [10]:
metadatas = [
    {"topic": "AI"} for _ in documents
]

ids = [f"doc_{i+1}" for i in range(len(documents))]

In [11]:
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print("20 documents added successfully")
print("Collection count:", collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 62.0MiB/s]


20 documents added successfully
Collection count: 20


In [12]:
query = "How does artificial intelligence learn from data?"

results = collection.query(
    query_texts=[query],
    n_results=5
)

print("Query:", query)
print("\nTop 5 results:\n")

for i, doc in enumerate(results["documents"][0], 1):
    print(f"{i}. {doc}")

Query: How does artificial intelligence learn from data?

Top 5 results:

1. Machine learning allows computers to learn patterns from data.
2. Artificial intelligence enables machines to perform tasks that normally require human intelligence.
3. Natural language processing helps computers understand human language.
4. Deep learning uses neural networks with multiple layers.
5. ChromaDB is a vector database designed for AI applications.


In [13]:
filtered_results = collection.query(
    query_texts=["AI applications"],
    n_results=5,
    where={"topic": "AI"}
)

print("Metadata-filtered results:\n")

for i, doc in enumerate(filtered_results["documents"][0], 1):
    print(f"{i}. {doc}")

Metadata-filtered results:

1. Artificial intelligence enables machines to perform tasks that normally require human intelligence.
2. AI applications can use embeddings for efficient information retrieval.
3. ChromaDB is a vector database designed for AI applications.
4. Machine learning allows computers to learn patterns from data.
5. Natural language processing helps computers understand human language.


In [14]:
print("Manual verification:")
print("- Query: AI applications")
print("- Filter: topic = AI")
print("- Retrieved documents:", len(filtered_results["documents"][0]))

for doc in filtered_results["documents"][0]:
    print("✓", doc)

Manual verification:
- Query: AI applications
- Filter: topic = AI
- Retrieved documents: 5
✓ Artificial intelligence enables machines to perform tasks that normally require human intelligence.
✓ AI applications can use embeddings for efficient information retrieval.
✓ ChromaDB is a vector database designed for AI applications.
✓ Machine learning allows computers to learn patterns from data.
✓ Natural language processing helps computers understand human language.


In [15]:
from google.colab import files

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

print("Uploaded PDF:", pdf_path)

Saving W5D3_AI_ML_Retrieval_Document.pdf to W5D3_AI_ML_Retrieval_Document.pdf
Uploaded PDF: W5D3_AI_ML_Retrieval_Document.pdf


In [19]:
pdf_path = "your_file.pdf"

In [21]:
!pip install -q -U langchain-text-splitters

In [24]:
from google.colab import files

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

print("PDF selected:", pdf_path)

Saving W5D3_AI_ML_Retrieval_Document.pdf to W5D3_AI_ML_Retrieval_Document (1).pdf
PDF selected: W5D3_AI_ML_Retrieval_Document (1).pdf


In [25]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(pdf_path)

pages = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(pages)

print("Number of pages:", len(pages))
print("Number of chunks:", len(chunks))

Number of pages: 2
Number of chunks: 9


In [30]:
pdf_collection = client.create_collection(
    name="W5D3_AI_ML_Retrieval_Document.pdf",
    metadata={"hnsw:space": "cosine"}
)

pdf_documents = [chunk.page_content for chunk in chunks]
pdf_metadatas = [
    {"page": chunk.metadata.get("page", 0)}
    for chunk in chunks
]
pdf_ids = [f"chunk_{i+1}" for i in range(len(chunks))]

pdf_collection.add(
    documents=pdf_documents,
    metadatas=pdf_metadatas,
    ids=pdf_ids
)

print("PDF chunks added to ChromaDB")
print("Total chunks:", pdf_collection.count())

PDF chunks added to ChromaDB
Total chunks: 9


In [31]:
query = "What is the main topic discussed in this document?"

top_results = pdf_collection.query(
    query_texts=[query],
    n_results=3
)

print("Top 3 retrieved chunks:\n")

for i, chunk in enumerate(top_results["documents"][0], 1):
    print(f"\n--- Chunk {i} ---")
    print(chunk)

Top 3 retrieved chunks:


--- Chunk 1 ---
generate, summarize, classify, and answer questions about text. In a RAG system, an LLM can use retrieved
document chunks as context before generating an answer.
Purpose of this document
This document is provided as a sample PDF for the W5D3 ChromaDB practical. It can be used to
demonstrate PDF text extraction, document chunking, vector storage, similarity retrieval, and passing
retrieved context to a local Ollama language model.

--- Chunk 2 ---
Retrieval-Augmented Generation, or RAG, combines information retrieval with a language model. Documents
are divided into smaller chunks and stored in a vector database. When a user asks a question, the system
retrieves the most relevant chunks and provides them as context to a language model. This can help the
model answer questions using information from a specific document.
9. Vector Databases and ChromaDB

--- Chunk 3 ---
structures within the data. Clustering is a common example. It groups similar 

In [32]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(
    model="llama3.2:3b"
)

context = "\n\n".join(top_results["documents"][0])

prompt = f"""
Answer the question using only the provided context.

Context:
{context}

Question:
{query}

Answer:
"""

answer = llm.invoke(prompt)

print("Question:", query)
print("\nAnswer:\n")
print(answer)

Question: What is the main topic discussed in this document?

Answer:

The main topic discussed in this document appears to be Retrieval-Augmented Generation (RAG), a system that combines information retrieval with a language model.


In [33]:
print("RAG Verification")
print("=" * 50)

print("Retrieved chunks:", len(top_results["documents"][0]))
print("\nGenerated answer:")
print(answer)

print("\n✓ ChromaDB retrieved the top-3 relevant chunks.")
print("✓ Ollama generated an answer using the retrieved context.")

RAG Verification
Retrieved chunks: 3

Generated answer:
The main topic discussed in this document appears to be Retrieval-Augmented Generation (RAG), a system that combines information retrieval with a language model.

✓ ChromaDB retrieved the top-3 relevant chunks.
✓ Ollama generated an answer using the retrieved context.
